## The goal
Bulk-compute the same 779-dim user feature vector as `build_user_feature.ipynb`, but for **every UCSD train-split user**. Output:

- `data/transformed/train_user_features.npy` — shape `(n_users, 779)`
- `data/transformed/user_id_to_index.json` — UCSD `user_id` (string) → row index

Same recipe as step 13:
- like_emb (384) — rating-weighted mean of embeddings for 4+ star books
- dislike_emb (384) — simple mean of embeddings for 1-2 star books (zero if none)
- genre_dist (10) — L2-normalized sum of genre vectors of 4+ star books
- mean_pages (1) — mean of normalized page values across 4+ star books

Two important constraints:
1. **Train-split only.** Features for *every* user (train/val/test) come from train-split interactions. Val and test interactions are never seen at feature time — that's the no-leakage rule.
2. **Drop users with no liked books.** They can't generate positive training pairs, so they contribute nothing to training and shouldn't bloat the matrix.

Implementation note: with ~250K users × 1.78M books, the interaction grid is 99.99% sparse. Sparse matrix multiplication does the four aggregations vectorized — runs in seconds, not minutes.

In [1]:
import json
import numpy as np
import polars as pl
from scipy.sparse import csr_matrix
from mybookrec import DATA_DIR

transformed = DATA_DIR / "transformed"

# Item-side artifacts (precomputed in steps 11-12). Cast embeddings to float32 so sparse matmul
# behaves; float16 isn't widely supported by scipy.sparse and is brittle for big sums.
with open(transformed / "book_id_to_index.json", "r") as f:
    book_id_to_index = json.load(f)

book_embeddings = np.load(transformed / "book_embeddings.npy").astype(np.float32)
genre_matrix = np.load(transformed / "genre_matrix.npy").astype(np.float32)
pages_vec = np.load(transformed / "num_pages_normalized.npy").astype(np.float32)

n_books = book_embeddings.shape[0]
embedding_dim = book_embeddings.shape[1]
n_genres = genre_matrix.shape[1]
print(f"Books: {n_books:,} | embedding_dim: {embedding_dim} | n_genres: {n_genres}")

Books: 1,782,579 | embedding_dim: 384 | n_genres: 10


In [2]:
# Stream the interaction parquet, projecting only the columns we need.
# books_with_interactions.parquet is ~19GB on disk because it has every book column duplicated
# per interaction; with column projection it shrinks to ~1.5GB in memory.
# Filter to the train split — feature construction uses train interactions only.
interactions = (
    pl.scan_parquet(transformed / "books_with_interactions.parquet")
    .filter(pl.col("data_split") == "train")
    .select("user_id", "book_id", "rating")
    .collect()
)
print(f"Train interactions: {len(interactions):,}")
print(interactions.head(3))

Train interactions: 72,497,704
shape: (3, 3)
┌─────────────────────────────────┬──────────┬────────┐
│ user_id                         ┆ book_id  ┆ rating │
│ ---                             ┆ ---      ┆ ---    │
│ str                             ┆ str      ┆ f64    │
╞═════════════════════════════════╪══════════╪════════╡
│ f8a89075dc6de14857561522e729f8… ┆ 8376729  ┆ 5.0    │
│ f8a89075dc6de14857561522e729f8… ┆ 13124276 ┆ 4.0    │
│ f8a89075dc6de14857561522e729f8… ┆ 12604778 ┆ 4.0    │
└─────────────────────────────────┴──────────┴────────┘


In [3]:
# Map book_id (string) -> row index in the item-side matrices.
# Drop rows whose book_id isn't in the catalog (shouldn't happen given the inner-join in transform_and_save,
# but guard anyway — a stray miss here would index out of bounds in the sparse matrix build).
interactions = (
    interactions
    .with_columns(pl.col("book_id").cast(pl.String))
    .with_columns(
        pl.col("book_id").map_elements(
            lambda b: book_id_to_index.get(b, -1), return_dtype=pl.Int64
        ).alias("book_idx")
    )
    .filter(pl.col("book_idx") >= 0)
)

# Assign integer user indices. user_id in the parquet is a string hash (e.g. '8842281e1d...').
unique_users = interactions["user_id"].unique().to_list()
user_id_to_index = {uid: i for i, uid in enumerate(unique_users)}
interactions = interactions.with_columns(
    pl.col("user_id").map_elements(
        lambda u: user_id_to_index[u], return_dtype=pl.Int64
    ).alias("user_idx")
)

n_users = len(unique_users)
print(f"Unique train users: {n_users:,}")
print(f"Interactions after book lookup: {len(interactions):,}")

Unique train users: 788,239
Interactions after book lookup: 72,497,704


In [4]:
# Build three sparse (n_users, n_books) matrices:
#   W_like[u, b]    = rating  if u rated b with rating >= 4, else 0  (rating-weighted likes)
#   M_like[u, b]    = 1       if u rated b with rating >= 4, else 0  (binary likes for genre/pages)
#   M_dislike[u, b] = 1       if u rated b with rating <= 2, else 0  (binary dislikes)
# 3-star ratings appear in neither — they're excluded as ambiguous taste signal.
user_idx = interactions["user_idx"].to_numpy()
book_idx = interactions["book_idx"].to_numpy()
rating = interactions["rating"].to_numpy().astype(np.float32)

liked_mask_arr = rating >= 4
disliked_mask_arr = rating <= 2

W_like = csr_matrix(
    (rating[liked_mask_arr], (user_idx[liked_mask_arr], book_idx[liked_mask_arr])),
    shape=(n_users, n_books),
)
M_like = csr_matrix(
    (np.ones(liked_mask_arr.sum(), dtype=np.float32),
     (user_idx[liked_mask_arr], book_idx[liked_mask_arr])),
    shape=(n_users, n_books),
)
M_dislike = csr_matrix(
    (np.ones(disliked_mask_arr.sum(), dtype=np.float32),
     (user_idx[disliked_mask_arr], book_idx[disliked_mask_arr])),
    shape=(n_users, n_books),
)

print(f"W_like nnz:    {W_like.nnz:,}")
print(f"M_like nnz:    {M_like.nnz:,}")
print(f"M_dislike nnz: {M_dislike.nnz:,}")

W_like nnz:    50,971,385
M_like nnz:    50,971,385
M_dislike nnz: 5,660,885


In [5]:
# Like and dislike embeddings via sparse matmul.
# (W_like @ book_embeddings) sums rating-weighted embedding rows per user.
# Dividing by per-user total weight gives the weighted mean.
like_weight_sum = np.asarray(W_like.sum(axis=1)).flatten()        # (n_users,)
dislike_count = np.asarray(M_dislike.sum(axis=1)).flatten()        # (n_users,)
liked_count = np.asarray(M_like.sum(axis=1)).flatten()             # (n_users,)

# Guard divisions: replace 0 with 1 so the division yields 0 (numerator is also 0 for those rows).
safe_like_weight = np.where(like_weight_sum > 0, like_weight_sum, 1.0).astype(np.float32)
safe_dislike_count = np.where(dislike_count > 0, dislike_count, 1.0).astype(np.float32)

like_emb = (W_like @ book_embeddings) / safe_like_weight[:, None]
dislike_emb = (M_dislike @ book_embeddings) / safe_dislike_count[:, None]

print(f"like_emb shape:    {like_emb.shape}")
print(f"dislike_emb shape: {dislike_emb.shape}")
print(f"Users with no disliked books: {int((dislike_count == 0).sum()):,}")
print(f"Users with no liked books:    {int((liked_count == 0).sum()):,}  (will be dropped)")

like_emb shape:    (788239, 384)
dislike_emb shape: (788239, 384)
Users with no disliked books: 281,936
Users with no liked books:    5,207  (will be dropped)


In [6]:
# Genre distribution: sum unweighted genre vectors of liked books, then L2-normalize per user.
# Mean pages: average of normalized page values over liked books.
genre_unnorm = M_like @ genre_matrix                              # (n_users, 10)
genre_norms = np.linalg.norm(genre_unnorm, axis=1, keepdims=True)
safe_genre_norms = np.where(genre_norms > 0, genre_norms, 1.0).astype(np.float32)
genre_dist = genre_unnorm / safe_genre_norms

safe_liked_count = np.where(liked_count > 0, liked_count, 1.0).astype(np.float32)
mean_pages_unnorm = M_like @ pages_vec.reshape(-1, 1)              # (n_users, 1)
mean_pages = (mean_pages_unnorm.flatten() / safe_liked_count).astype(np.float32)

print(f"genre_dist shape: {genre_dist.shape}")
print(f"mean_pages shape: {mean_pages.shape}")

genre_dist shape: (788239, 10)
mean_pages shape: (788239,)


In [7]:
# Filter to valid users (at least 1 liked book) and compact the index mapping.
valid_mask = liked_count > 0
n_valid = int(valid_mask.sum())

user_features = np.concatenate([
    like_emb[valid_mask],                       # (n_valid, 384)
    dislike_emb[valid_mask],                    # (n_valid, 384)
    genre_dist[valid_mask],                     # (n_valid, 10)
    mean_pages[valid_mask].reshape(-1, 1),      # (n_valid, 1)
], axis=1).astype(np.float32)

expected_dim = embedding_dim * 2 + n_genres + 1
assert user_features.shape == (n_valid, expected_dim), f"Expected ({n_valid}, {expected_dim}), got {user_features.shape}"
assert not np.isnan(user_features).any(), "NaNs in user_features — check guard divisions"

# Compact mapping: original user_idx -> new row in the filtered matrix.
valid_old_indices = np.where(valid_mask)[0]
compact_user_id_to_index = {
    unique_users[old_idx]: new_idx
    for new_idx, old_idx in enumerate(valid_old_indices)
}

np.save(transformed / "train_user_features.npy", user_features)
with open(transformed / "user_id_to_index.json", "w") as f:
    json.dump(compact_user_id_to_index, f)

print(f"Saved train_user_features of shape {user_features.shape}")
print(f"Saved user_id_to_index with {len(compact_user_id_to_index):,} entries")
print(f"Dropped users with no liked books: {n_users - n_valid:,}")

Saved train_user_features of shape (783032, 779)
Saved user_id_to_index with 783,032 entries
Dropped users with no liked books: 5,207


In [8]:
# Sanity checks on the saved matrix — useful diagnostics before training.
like_norms = np.linalg.norm(user_features[:, :embedding_dim], axis=1)
dislike_norms = np.linalg.norm(user_features[:, embedding_dim:2*embedding_dim], axis=1)
genre_norms_check = np.linalg.norm(user_features[:, 2*embedding_dim:2*embedding_dim+n_genres], axis=1)
mean_pages_col = user_features[:, -1]

print(f"like_emb    norm: mean={like_norms.mean():.3f}  std={like_norms.std():.3f}  min={like_norms.min():.3f}  max={like_norms.max():.3f}")
print(f"dislike_emb norm: mean={dislike_norms.mean():.3f}  std={dislike_norms.std():.3f}  zero={(dislike_norms == 0).sum():,} (no disliked books)")
print(f"genre_dist  norm: mean={genre_norms_check.mean():.3f}  (should be ≈1.0 for users with genre data)")
print(f"mean_pages: mean={mean_pages_col.mean():.3f}  std={mean_pages_col.std():.3f}  range=[{mean_pages_col.min():.3f}, {mean_pages_col.max():.3f}]")

like_emb    norm: mean=0.815  std=0.046  min=0.752  max=1.001
dislike_emb norm: mean=0.563  std=0.424  zero=279,304 (no disliked books)
genre_dist  norm: mean=0.999  (should be ≈1.0 for users with genre data)
mean_pages: mean=0.446  std=0.101  range=[0.000, 1.000]
